# KG1 V206B H100 Answer-Only Loss-Gated Colab

Purpose: execute the next conservative roadmap step for the NVIDIA Nemotron Kaggle challenge.

This notebook:

- clones `FELIPEACASTRO/KG1-NVIDIA` branch `claude/competent-shamir`;
- requires the exact V194 rank-19 baseline adapter by SHA256;
- validates the curated V206 dataset and V198 strict validation split by SHA256;
- trains one conservative continuation candidate on H100/A100 80GB with `MAX_LENGTH=8192`;
- requires final validation loss to be no worse than baseline before packaging;
- runs structural Kaggle adapter packaging gates;
- never submits to Kaggle.

V206B is a response-objective correction after V206A regressed: it trains only concise `\boxed{answer}` completions from a 1680-row balanced micro dataset, with one `1e-9` update before the same no-regression gate.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import re
import shutil
import subprocess
import sys
import zipfile

VERSION = 'V206B_H100_ANSWER_ONLY_LOSS_GATED_20260506'
print('NOTEBOOK_VERSION =', VERSION)

REPO_URL = os.environ.get('KG1_REPO_URL', 'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git')
REPO_BRANCH = os.environ.get('KG1_REPO_BRANCH', 'claude/competent-shamir')
ROOT = pathlib.Path('/content/kg1')
SCRIPT_DIR = ROOT / 'scripts'
TRAIN_SCRIPT = SCRIPT_DIR / 'hf_job_train_v90.py'
POSTTRAIN_GATE = SCRIPT_DIR / 'kg1_v202d_posttrain_gate.py'
PREFLIGHT_SCRIPT = SCRIPT_DIR / 'nemotron_submission_preflight.py'

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V206B')
DRIVE_V202D = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V202D')
RANK19_BUILD = DRIVE_V202D / 'init_adapter_v194_rank19_build'
INIT_ADAPTER = RANK19_BUILD / 'adapter'
OUT_ROOT = DRIVE_ROOT / 'output_v206b_answer_only_h100_loss_gated'
DRY_RUN_OUT = OUT_ROOT / 'dry_run_validate'
TRAIN_OUT = OUT_ROOT / 'train_v206b_answer_only_1s_lr1e9'
REPORT_DIR = OUT_ROOT / 'reports'

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = 'cbd3fa9f933d55ef16a84236559f4ee2a0526848'
V194_RANK19_ZIP_SHA256 = '49886191bf9ce92a48106ebfcba407bf9edbe423a4ed8c476d1f6bdfdd210fd8'
V194_RANK19_ADAPTER_MODEL_SHA256 = '01259fef943bc16c31d8f7907be076cc987381a6a1bbe732b1b33c2d9f2ea95f'
V194_RANK19_ADAPTER_CONFIG_SHA256 = 'e5499f128fde60d32d0595d427e4fe84d8abe6dbde1d80886c970e8184e4b743'
V206B_TRAIN_SHA256 = '5c95e9b254a3b3a37850db1c4f75914d4b23233f0cd1e9d828886399e9a42f5d'
V198_VAL_SHA256 = 'e59c907c6545e5e587097a64762e3e874508e8cd74d85d5c7c79354ebe56e73c'

TRAIN_FILE = ROOT / 'data/v206b/v206b_answer_only_micro_train.jsonl'
VAL_FILE = ROOT / 'data/v198/v198_micro_val.strict.jsonl'
MANIFEST_FILE = ROOT / 'data/v206b/v206b_answer_only_manifest.json'

RUN_DRY_RUN_VALIDATE = True
RUN_TRAIN = True
RUN_PACKAGE = True
ALLOW_KAGGLE_SUBMIT = False

for path in [DRIVE_ROOT, OUT_ROOT, DRY_RUN_OUT, TRAIN_OUT, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('REPO_URL =', REPO_URL)
print('REPO_BRANCH =', REPO_BRANCH)
print('OUT_ROOT =', OUT_ROOT)
print('TRAIN_OUT =', TRAIN_OUT)
print('ALLOW_KAGGLE_SUBMIT =', ALLOW_KAGGLE_SUBMIT)
if ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('This notebook is intentionally submit-disabled.')


In [ ]:
def run(cmd, cwd=None, env=None, check=True):
    cmd = [str(part) for part in cmd]
    print('+', ' '.join(cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, check=check)


def stream_process(cmd, cwd=None, env=None, log_path=None):
    cmd = [str(part) for part in cmd]
    print('+', ' '.join(cmd))
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_path.open('w', encoding='utf-8')
    else:
        log_handle = None
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        if log_handle:
            log_handle.write(line)
    rc = proc.wait()
    if log_handle:
        log_handle.close()
    print('returncode =', rc)
    return rc


def sha256_path(path: pathlib.Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def assert_sha256(path: pathlib.Path, expected: str, label: str) -> str:
    observed = sha256_path(path)
    print(f'{label} sha256:', observed)
    if observed.lower() != expected.lower():
        raise RuntimeError(f'{label} SHA mismatch for {path}: {observed} != {expected}')
    return observed


def write_json(path: pathlib.Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')


def read_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value or ''
    except Exception:
        return ''


In [ ]:
gpu_csv = subprocess.check_output(
    'nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader,nounits',
    shell=True,
).decode().strip()
print('GPU:', gpu_csv)
parts = [part.strip() for part in gpu_csv.split(',')]
gpu_name = parts[0]
gpu_mem_mib = int(parts[1])
driver_version = parts[2]
assert ('H100' in gpu_name) or ('A100' in gpu_name and gpu_mem_mib >= 75000), (
    f'Use H100 or A100 80GB High-RAM; found {gpu_name} with {gpu_mem_mib} MiB.'
)
meminfo = pathlib.Path('/proc/meminfo').read_text(encoding='utf-8')
host_mem_kib = int(re.search(r'MemTotal:\s+(\d+)', meminfo).group(1))
host_mem_gib = host_mem_kib / 1024 / 1024
disk_free_gib = shutil.disk_usage('/content').free / 1024**3
print(f'Host RAM: {host_mem_gib:.1f} GiB')
print(f'/content free: {disk_free_gib:.1f} GiB')
print('Driver:', driver_version)
assert host_mem_gib >= 50, f'High-RAM runtime expected; host RAM is {host_mem_gib:.1f} GiB'
assert disk_free_gib >= 90, f'Need at least 90 GiB free on /content; found {disk_free_gib:.1f} GiB'


In [ ]:
if ROOT.exists():
    run(['git', '-C', ROOT, 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', ROOT, 'checkout', REPO_BRANCH])
    run(['git', '-C', ROOT, 'pull', '--ff-only', 'origin', REPO_BRANCH])
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, ROOT])

commit = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD']).decode().strip()
print('Repo commit:', commit)
for required in [TRAIN_SCRIPT, POSTTRAIN_GATE, PREFLIGHT_SCRIPT, TRAIN_FILE, VAL_FILE, MANIFEST_FILE]:
    if not required.exists():
        raise FileNotFoundError(required)
assert_sha256(TRAIN_FILE, V206B_TRAIN_SHA256, 'V206B answer-only train')
assert_sha256(VAL_FILE, V198_VAL_SHA256, 'V198 strict validation')
manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
print('V206B answer-only train rows:', manifest.get('train_rows'))
print('V206 family counts:', manifest.get('family_counts'))


In [ ]:
import importlib.metadata as md
import importlib.util

os.environ.setdefault('MAX_JOBS', '4')
os.environ.setdefault('PIP_ROOT_USER_ACTION', 'ignore')


def pip_install(args):
    print('+ pip install', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])


def pip_uninstall(package_name):
    print('+ pip uninstall -y', package_name)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', package_name], check=False)


def install_exact(dist_name, expected_version, module_name, args):
    try:
        observed = md.version(dist_name)
    except md.PackageNotFoundError:
        observed = None
    if observed != expected_version:
        print(f'{dist_name} version {observed!r}; installing {expected_version}')
        pip_install(args)
    else:
        print(f'{dist_name} already at {expected_version}')
    assert importlib.util.find_spec(module_name) is not None, f'{module_name} import spec missing after install'
    assert md.version(dist_name) == expected_version, f'{dist_name} version mismatch after install: {md.version(dist_name)}'


pip_uninstall('torchao')
pip_install(['--upgrade', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja==1.13.0'])
pip_install([
    'kaggle==2.0.2',
    'transformers==5.7.0',
    'accelerate==1.13.0',
    'peft==0.19.1',
    'datasets==4.8.5',
    'safetensors==0.7.0',
    'huggingface_hub==1.13.0',
    'sentencepiece==0.2.1',
    'protobuf==7.34.1',
])
install_exact('causal-conv1d', '1.6.1', 'causal_conv1d', ['causal-conv1d==1.6.1', '--no-build-isolation'])
install_exact('mamba-ssm', '2.3.1', 'mamba_ssm', ['mamba-ssm==2.3.1', '--no-build-isolation'])
assert importlib.util.find_spec('torchao') is None, 'torchao still installed; restart runtime and rerun cells from top'
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
print('mamba_ssm OK:', getattr(mamba_ssm, '__version__', 'unknown'))


In [ ]:
hf_token = os.environ.get('HF_TOKEN') or read_colab_secret('HF_TOKEN') or read_colab_secret('HUGGINGFACE_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN staged from env/Colab secrets; value is not printed.')
else:
    print('HF_TOKEN not found. If the NVIDIA model is gated for this account, add HF_TOKEN as a Colab secret.')


In [ ]:
def adapter_ready(path: pathlib.Path) -> bool:
    cfg = path / 'adapter_config.json'
    model = path / 'adapter_model.safetensors'
    return cfg.exists() and model.exists()


def extract_v194_zip(candidate: pathlib.Path) -> pathlib.Path:
    safe_source = OUT_ROOT / 'source_v194_rank19_submission.zip'
    safe_source.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidate, safe_source)
    assert_sha256(safe_source, V194_RANK19_ZIP_SHA256, 'V194 rank-19 submission.zip')
    if RANK19_BUILD.exists():
        shutil.rmtree(RANK19_BUILD)
    INIT_ADAPTER.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(safe_source) as zf:
        members = {name for name in zf.namelist() if not name.endswith('/')}
        expected = {'adapter_model.safetensors', 'adapter_config.json'}
        if members != expected:
            raise RuntimeError(f'Unexpected V194 zip members: {sorted(members)}')
        zf.extract('adapter_model.safetensors', INIT_ADAPTER)
        zf.extract('adapter_config.json', INIT_ADAPTER)
    cached_zip = RANK19_BUILD / 'submission.zip'
    shutil.copy2(safe_source, cached_zip)
    assert_sha256(INIT_ADAPTER / 'adapter_model.safetensors', V194_RANK19_ADAPTER_MODEL_SHA256, 'V194 adapter_model')
    assert_sha256(INIT_ADAPTER / 'adapter_config.json', V194_RANK19_ADAPTER_CONFIG_SHA256, 'V194 adapter_config')
    assert_sha256(cached_zip, V194_RANK19_ZIP_SHA256, 'cached V194 submission.zip')
    return INIT_ADAPTER


def ensure_v194_adapter() -> pathlib.Path:
    cached_zip = RANK19_BUILD / 'submission.zip'
    if adapter_ready(INIT_ADAPTER) and cached_zip.exists():
        assert_sha256(INIT_ADAPTER / 'adapter_model.safetensors', V194_RANK19_ADAPTER_MODEL_SHA256, 'cached V194 adapter_model')
        assert_sha256(INIT_ADAPTER / 'adapter_config.json', V194_RANK19_ADAPTER_CONFIG_SHA256, 'cached V194 adapter_config')
        assert_sha256(cached_zip, V194_RANK19_ZIP_SHA256, 'cached V194 submission.zip')
        return INIT_ADAPTER

    candidates = []
    if os.environ.get('V194_RANK19_ZIP'):
        candidates.append(pathlib.Path(os.environ['V194_RANK19_ZIP']))
    candidates += [
        DRIVE_V202D / 'init_adapter_v194_rank19_build/submission.zip',
        DRIVE_V202D / 'baseline_v194_rank19/submission.zip',
        pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V202/baseline_v194_rank19/submission.zip'),
        pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V199B/baseline_v194_rank19/submission.zip'),
    ]
    for candidate in candidates:
        if candidate.exists():
            try:
                return extract_v194_zip(candidate)
            except RuntimeError as exc:
                print('Skipping non-matching V194 candidate:', candidate, exc)
    paths = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(
        'Exact V194 rank-19 baseline submission.zip not found. '
        f'Expected SHA256 {V194_RANK19_ZIP_SHA256}. Checked:\n{paths}'
    )


ensure_v194_adapter()
print('V194 adapter ready:', INIT_ADAPTER)


In [ ]:
COMMON_TRAIN_ENV = {
    'MODEL_NAME': MODEL_NAME,
    'MODEL_REVISION': MODEL_REVISION,
    'MODEL_DEVICE_MAP': 'auto',
    'DATA_FILE': str(TRAIN_FILE),
    'VAL_FILE': str(VAL_FILE),
    'EXPECTED_TRAIN_SHA256': V206B_TRAIN_SHA256,
    'EXPECTED_VAL_SHA256': V198_VAL_SHA256,
    'MIN_TRAIN_EXAMPLES': '1680',
    'MIN_VAL_EXAMPLES': '720',
    'MIN_TOKENIZED_TRAIN_EXAMPLES': '1600',
    'MIN_TOKENIZED_VAL_EXAMPLES': '720',
    'INIT_ADAPTER_DIR': str(INIT_ADAPTER),
    'INIT_ADAPTER_LOAD_MODE': 'manual',
    'PEFT_MANUAL_LOAD_METHOD': 'direct',
    'ADAPTER_LOAD_LOW_CPU_MEM_USAGE': '0',
    'UPLOAD_TO_HF': '0',
    'UPLOAD_CHECKPOINTS_DURING_TRAINING': '0',
    'FAIL_ON_MISSING_ADAPTER_KEYS': '1',
    'REQUIRE_OFFSET_MASK': '1',
    'LORA_R': '32',
    'LORA_ALPHA': '32',
    'LORA_DROPOUT': '0.0',
    'LORA_TARGET_MODULES': 'down_proj,in_proj,k_proj,lm_head,o_proj,out_proj,q_proj,up_proj,v_proj',
    'TRAINABLE_LORA_MODULES': 'in_proj,out_proj,q_proj,k_proj,v_proj,o_proj',
    'MAX_TRAINABLE_PARAM_RATIO': '0.035',
    'MAX_LENGTH': '8192',
    'BATCH_SIZE': '1',
    'MICRO_BATCH_SIZE': '1',
    'NUM_EPOCHS': '1',
    'MAX_STEPS': '1',
    'SAVE_EVERY_STEPS': '1',
    'EVAL_EVERY_STEPS': '1',
    'EVAL_MAX_EXAMPLES': '720',
    'LOG_EVERY_STEPS': '1',
    'MICRO_LOG_EVERY': '0',
    'SEED': '2062',
    'SAMPLING_MODE': 'shuffle',
    'LEARNING_RATE': '1e-9',
    'FINAL_LEARNING_RATE': '1e-9',
    'GRAD_CLIP_NORM': '1.0',
    'BASELINE_EVAL_BEFORE_TRAIN': '1',
    'REQUIRE_FINAL_EVAL_LTE_BASELINE': '1',
    'MAX_FINAL_EVAL_REGRESSION': '0.0',
    'ABORT_EVAL_RELATIVE_TO_BASELINE_DELTA': '0.001',
    'ABORT_MAX_RESERVED_GIB': '79',
    'COMPUTE_PROVIDER': 'colab_v206b_answer_only_h100_loss_gated',
}
if os.environ.get('HF_TOKEN'):
    COMMON_TRAIN_ENV['HF_TOKEN'] = os.environ['HF_TOKEN']

contract = {
    'version': VERSION,
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat(),
    'repo': {'url': REPO_URL, 'branch': REPO_BRANCH},
    'baseline': {
        'label': 'V194_rank19_public_0.86',
        'zip_sha256': V194_RANK19_ZIP_SHA256,
        'adapter_model_sha256': V194_RANK19_ADAPTER_MODEL_SHA256,
    },
    'dataset': {
        'train_file': str(TRAIN_FILE),
        'train_sha256': V206B_TRAIN_SHA256,
        'validation_file': str(VAL_FILE),
        'validation_sha256': V198_VAL_SHA256,
        'manifest': manifest,
    },
    'env_contract': {k: ('<redacted>' if k == 'HF_TOKEN' else v) for k, v in COMMON_TRAIN_ENV.items()},
    'policy': {
        'allow_kaggle_submit': ALLOW_KAGGLE_SUBMIT,
        'require_final_eval_lte_baseline': True,
        'max_final_eval_regression': 0.0,
    },
}
write_json(REPORT_DIR / 'v206b_execution_contract.json', contract)
print(json.dumps(contract['env_contract'], indent=2, sort_keys=True))


In [ ]:
if RUN_DRY_RUN_VALIDATE:
    env = os.environ.copy()
    env.update(COMMON_TRAIN_ENV)
    env.update({
        'OUTPUT_DIR': str(DRY_RUN_OUT),
        'RUN_ID': 'v206b-dryrun-answer-only-8192',
        'DRY_RUN_VALIDATE_ONLY': '1',
        'BASELINE_EVAL_BEFORE_TRAIN': '0',
        'REQUIRE_FINAL_EVAL_LTE_BASELINE': '0',
        'EVAL_MAX_EXAMPLES': '96',
        'MAX_STEPS': '1',
    })
    rc = stream_process(
        [sys.executable, TRAIN_SCRIPT],
        cwd=ROOT,
        env=env,
        log_path=DRY_RUN_OUT / 'dry_run_validate.log',
    )
    if rc != 0:
        raise RuntimeError(f'Dry-run validation failed; see {DRY_RUN_OUT / "dry_run_validate.log"}')
    dry_report = DRY_RUN_OUT / 'dry_run_model_recipe_report.json'
    report = json.loads(dry_report.read_text(encoding='utf-8'))
    assert report['data']['train_records'] == 1680
    assert report['data']['validation_records'] == 720
    assert report['data']['tokenized_train_records'] >= 1600
    assert report['data']['tokenized_validation_records'] >= 720
    assert report['training']['max_length'] == 8192
    assert report['lora']['trainable_lora_module_filter']['enabled'] is True
    print('V206B dry-run passed:', dry_report)
else:
    print('RUN_DRY_RUN_VALIDATE=False; skipping dry-run.')


In [ ]:
train_summary = None
if RUN_TRAIN:
    env = os.environ.copy()
    env.update(COMMON_TRAIN_ENV)
    env.update({
        'OUTPUT_DIR': str(TRAIN_OUT),
        'RUN_ID': 'v206b-answer-only-1s-lr1e9',
        'DRY_RUN_VALIDATE_ONLY': '0',
    })
    rc = stream_process(
        [sys.executable, TRAIN_SCRIPT],
        cwd=ROOT,
        env=env,
        log_path=TRAIN_OUT / 'train_v206b_answer_only_1s_lr1e9.log',
    )
    manifest_path = TRAIN_OUT / 'final_adapter/v90_training_manifest.json'
    train_summary = {
        'returncode': rc,
        'manifest_path': str(manifest_path),
        'passed_no_regression_gate': False,
    }
    if manifest_path.exists():
        train_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        gate = train_manifest.get('training', {}).get('baseline_gate', {})
        baseline = gate.get('baseline_eval_loss')
        final = gate.get('final_eval_loss')
        train_summary.update({
            'baseline_eval_loss': baseline,
            'final_eval_loss': final,
            'delta_vs_baseline': None if baseline is None or final is None else round(final - baseline, 9),
            'passed_no_regression_gate': bool(rc == 0 and baseline is not None and final is not None and final <= baseline),
        })
    write_json(REPORT_DIR / 'v206b_train_summary.json', train_summary)
    print(json.dumps(train_summary, indent=2, sort_keys=True))
    if rc != 0:
        raise RuntimeError(f'V206B training failed or gate blocked; see {TRAIN_OUT}')
    if not train_summary['passed_no_regression_gate']:
        raise RuntimeError('V206B did not pass final-eval <= baseline gate; packaging is blocked.')
else:
    print('RUN_TRAIN=False; skipping training.')


In [ ]:
package_report = None
if RUN_PACKAGE:
    if not (TRAIN_OUT / 'final_adapter').exists():
        raise FileNotFoundError(TRAIN_OUT / 'final_adapter')
    posttrain_log = OUT_ROOT / 'v206b_posttrain_gate.log'
    rc = stream_process(
        [
            sys.executable,
            POSTTRAIN_GATE,
            '--root', ROOT,
            '--output-root', TRAIN_OUT,
            '--candidate-label', 'V206B_answer_only_1s_lr1e9',
            '--fail-on-block',
        ],
        cwd=ROOT,
        env=os.environ.copy(),
        log_path=posttrain_log,
    )
    if rc != 0:
        raise RuntimeError(f'Posttrain packaging gate failed; see {posttrain_log}')
    posttrain_report_path = TRAIN_OUT / 'posttrain_kaggle_gate_v202d/v202d_posttrain_gate_report.json'
    posttrain_report = json.loads(posttrain_report_path.read_text(encoding='utf-8'))
    primary_zip = pathlib.Path(posttrain_report['decision']['primary_zip'])
    preflight_json = OUT_ROOT / 'v206b_submission_preflight.json'
    preflight_log = OUT_ROOT / 'v206b_submission_preflight.log'
    rc = stream_process(
        [
            sys.executable,
            PREFLIGHT_SCRIPT,
            '--adapter-zip', str(primary_zip),
            '--output-json', str(preflight_json),
            '--fail-on-block',
        ],
        cwd=ROOT,
        env=os.environ.copy(),
        log_path=preflight_log,
    )
    if rc != 0:
        raise RuntimeError(f'Preflight failed for {primary_zip}; see {preflight_log}')
    package_report = {
        'posttrain_report_path': str(posttrain_report_path),
        'primary_zip': str(primary_zip),
        'primary_zip_sha256': sha256_path(primary_zip),
        'preflight_json': str(preflight_json),
        'do_not_submit_without_explicit_authorization': True,
    }
    write_json(REPORT_DIR / 'v206b_package_summary.json', package_report)
    print(json.dumps(package_report, indent=2, sort_keys=True))
else:
    print('RUN_PACKAGE=False; skipping packaging.')


In [ ]:
final_summary = {
    'version': VERSION,
    'output_root': str(OUT_ROOT),
    'train_summary': train_summary,
    'package_report': package_report,
    'no_kaggle_submit_performed': True,
    'next_gate_required_before_submission': [
        'review v206b_train_summary.json',
        'compare against V194/V199B Drive logs',
        'run solve-rate/vLLM proxy gate if available before any Kaggle upload',
    ],
}
write_json(OUT_ROOT / 'V206B_FINAL_RUN_SUMMARY.json', final_summary)
print(json.dumps(final_summary, indent=2, sort_keys=True))
print('FINAL_SUMMARY_PATH =', OUT_ROOT / 'V206B_FINAL_RUN_SUMMARY.json')
if package_report:
    print('CANDIDATE_ZIP =', package_report['primary_zip'])
print('KAGGLE_SUBMIT = BLOCKED_IN_THIS_NOTEBOOK')
